<center><font size="6" color="#2D6A4F"><b>CNN Training for Speckle Pattern Analysis -</b></font></center>
<center><font size="5" color="#2D6A4F"><b>Reproducing "Deep Speckle Correlation"</b></font></center>

In [ ]:
import os
import random
import time
import json
from pathlib import Path
from typing import Tuple

import numpy as np
import cv2
from matplotlib import pyplot as plt
from IPython.display import clear_output
from tqdm.keras import TqdmCallback
from sklearn.metrics import jaccard_score
from scipy.stats import pearsonr

import tensorflow as tf
import tensorflow.keras.backend as K
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import (Input, Conv2D, MaxPooling2D, UpSampling2D, Concatenate, Dropout, Activation, BatchNormalization)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, LearningRateScheduler
from tensorflow.keras.utils import to_categorical

seed_value = 11

os.environ['PYTHONHASHSEED'] = str(seed_value)
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'
os.environ['TF_DETERMINISTIC_OPS'] = '1'

random.seed(seed_value)
np.random.seed(seed_value)
tf.random.set_seed(seed_value)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

**Loading and Preprocessing Function for Training and Validation Sets**

In [ ]:
def load_processed_train_valid(dataset_name: str) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    base_path = Path('/content/drive/MyDrive/TranSpeckle/data_sets') / dataset_name

    def binning_2x2(image: np.ndarray) -> np.ndarray:
        """Performs 2x2 binning on the input image."""
        return image.reshape((image.shape[0] // 2, 2, image.shape[1] // 2, 2)).mean(axis=(1, 3))

    def process_pair(x_dir: Path, y_dir: Path, label: str) -> Tuple[np.ndarray, np.ndarray]:
        x_list = []
        y_list = []

        x_files = sorted(x_dir.glob('*.tif'))
        print(f"Found {len(x_files)} files in '{label}'")

        t0 = time.time()
        for i, x_file in enumerate(x_files):
            fname = x_file.name
            y_file = y_dir / fname

            if not y_file.exists():
                print(f"Missing GT for: {fname}")
                continue

            # Load the input image (grayscale)
            x = cv2.imread(str(x_file), cv2.IMREAD_GRAYSCALE)
            if x is None:
                print(f"Failed to load: {x_file}")
                continue

            x = binning_2x2(x)
            max_value = np.max(x)
            if max_value > 0:
                x = x / max_value  # normalize to [0,1]

            x = x.astype(np.float32)
            x = np.expand_dims(x, axis=-1)

            # Load the ground truth image (grayscale)
            y = cv2.imread(str(y_file), cv2.IMREAD_GRAYSCALE)
            if y is None:
                print(f"Failed to load GT: {y_file}")
                continue

            y = binning_2x2(y)
            y = (y > 0).astype(np.uint8)
            y = to_categorical(y, num_classes=2)

            x_list.append(x)
            y_list.append(y)

            if i % 100 == 0 and i > 0:
                print(f"Processed {i}/{len(x_files)} files in '{label}'")

        print(f"Done loading '{label}' in {time.time() - t0:.1f} seconds")
        return np.array(x_list), np.array(y_list)

    print("Starting to load and preprocess training and validation sets...")
    x_train, y_train = process_pair(base_path / 'x_train', base_path / 'y_train', 'train')
    x_valid, y_valid = process_pair(base_path / 'x_valid', base_path / 'y_valid', 'valid')

    print(f"Loaded all data from '{dataset_name}' — train: {x_train.shape[0]}, valid: {x_valid.shape[0]}")
    return x_train, y_train, x_valid, y_valid


**Loading and Preprocessing Function for the Test Set**

In [ ]:
def load_processed_test(dataset_name: str) -> Tuple[np.ndarray, np.ndarray]:
    base_path = Path('/content/drive/MyDrive/TranSpeckle/data_sets') / dataset_name
    x_path = base_path / 'x_test'
    y_path = base_path / 'y_test'

    x_list = []
    y_list = []

    def binning_2x2(image: np.ndarray) -> np.ndarray:
        """Performs 2x2 binning on the input image."""
        return image.reshape((image.shape[0] // 2, 2, image.shape[1] // 2, 2)).mean(axis=(1, 3))

    x_files = sorted(x_path.glob('*.tif'))
    print(f"\nStarting to load and preprocess test set from '{dataset_name}'")
    print(f"Found {len(x_files)} test files")

    t0 = time.time()
    for i, x_file in enumerate(x_files):
        fname = x_file.name
        y_file = y_path / fname

        if not y_file.exists():
            print(f"Missing GT for: {fname}")
            continue

        # Load the input speckle image (grayscale)
        x = cv2.imread(str(x_file), cv2.IMREAD_GRAYSCALE)
        if x is None:
            print(f"Failed to load speckle: {x_file}")
            continue

        x = binning_2x2(x)
        max_value = np.max(x)
        if max_value > 0:
            x = x / max_value  # normalize to [0,1]
        x = x.astype(np.float32)
        x = np.expand_dims(x, axis=-1)

        # Load the ground truth image (grayscale)
        y = cv2.imread(str(y_file), cv2.IMREAD_GRAYSCALE)
        if y is None:
            print(f"Failed to load GT: {y_file}")
            continue

        # Apply 2x2 binning on the ground truth as well
        y = binning_2x2(y)
        y = (y > 0).astype(np.uint8)
        y = to_categorical(y, num_classes=2)

        # Append processed images to the lists
        x_list.append(x)
        y_list.append(y)

        if i % 50 == 0 and i > 0:
            print(f"Processed {i}/{len(x_files)} files")

    print(f"Done loading test set in {time.time() - t0:.1f} seconds")
    print(f"Total loaded: {len(x_list)} examples")
    return np.array(x_list), np.array(y_list)


**CNN-U-Net with Dense Blocks model for Deep Speckle Correlation**

In [ ]:
def conv_factory(x, concat_axis, nb_filter,
                 dropout_rate=None, weight_decay=1E-4):
    x = BatchNormalization(axis=concat_axis,
                           gamma_regularizer=l2(weight_decay),
                           beta_regularizer=l2(weight_decay))(x)
    x = Activation('relu')(x)
    x = Conv2D(nb_filter, (5, 5), dilation_rate=(2, 2),
               kernel_initializer="he_uniform",
               padding="same",
               kernel_regularizer=l2(weight_decay))(x)
    if dropout_rate:
        x = Dropout(dropout_rate)(x)

    return x

# define dense block
def denseblock(x, concat_axis, nb_layers, growth_rate,
               dropout_rate=None, weight_decay=1E-4):
    list_feat = [x]
    for i in range(nb_layers):
        x = conv_factory(x, concat_axis, growth_rate,
                         dropout_rate, weight_decay)
        list_feat.append(x)
        x = Concatenate(axis=concat_axis)(list_feat)

    return x

# define model U-net modified with dense block
def get_model_deep_speckle():
    inputs = Input((256, 256, 1))
    print("inputs shape:", inputs.shape)

    conv1 = Conv2D(64, 3, activation='relu', padding='same', kernel_initializer='he_normal')(inputs)
    print("conv1 shape:", conv1.shape)
    db1 = denseblock(x=conv1, concat_axis=3, nb_layers=4, growth_rate=16, dropout_rate=0.5)
    print("db1 shape:", db1.shape)
    pool1 = MaxPooling2D(pool_size=(2, 2))(db1)
    print("pool1 shape:", pool1.shape)

    conv2 = Conv2D(128, 3, activation='relu', padding='same', kernel_initializer='he_normal')(pool1)
    print("conv2 shape:", conv2.shape)
    db2 = denseblock(x=conv2, concat_axis=3, nb_layers=4, growth_rate=16, dropout_rate=0.5)
    print("db2 shape:", db2.shape)
    pool2 = MaxPooling2D(pool_size=(2, 2))(db2)
    print("pool2 shape:", pool2.shape)

    conv3 = Conv2D(256, 3, activation='relu', padding='same', kernel_initializer='he_normal')(pool2)
    print("conv3 shape:", conv3.shape)
    db3 = denseblock(x=conv3, concat_axis=3, nb_layers=4, growth_rate=16, dropout_rate=0.5)
    print("db3 shape:", db3.shape)
    pool3 = MaxPooling2D(pool_size=(2, 2))(db3)
    print("pool3 shape:", pool3.shape)

    conv4 = Conv2D(512, 3, activation='relu', padding='same', kernel_initializer='he_normal')(pool3)
    print("conv4 shape:", conv4.shape)
    db4 = denseblock(x=conv4, concat_axis=3, nb_layers=4, growth_rate=16, dropout_rate=0.5)
    print("db4 shape:", db4.shape)
    pool4 = MaxPooling2D(pool_size=(2, 2))(db4)
    print("pool4 shape:", pool4.shape)

    conv5 = Conv2D(1024, 3, activation='relu', padding='same', kernel_initializer='he_normal')(pool4)
    print("conv5 shape:", conv5.shape)
    db5 = denseblock(x=conv5, concat_axis=3, nb_layers=4, growth_rate=16, dropout_rate=0.5)
    print("db5 shape:", db5.shape)
    up5 = Conv2D(512, 2, activation='relu', padding='same', kernel_initializer='he_normal')(
        UpSampling2D(size=(2, 2))(db5))
    print("up5 shape:", up5.shape)
    merge5 = Concatenate(axis=3)([db4, up5])
    print("merge5 shape:", merge5.shape)

    conv6 = Conv2D(512, 3, activation='relu', padding='same', kernel_initializer='he_normal')(merge5)
    print("conv6 shape:", conv6.shape)
    db6 = denseblock(x=conv6, concat_axis=3, nb_layers=3, growth_rate=16, dropout_rate=0.5)
    print("db5 shape:", db6.shape)
    up6 = Conv2D(256, 2, activation='relu', padding='same', kernel_initializer='he_normal')(
        UpSampling2D(size=(2, 2))(db6))
    print("up6 shape:", up6.shape)
    merge6 = Concatenate(axis=3)([db3, up6])
    print("merge6 shape:", merge6.shape)

    conv7 = Conv2D(256, 3, activation='relu', padding='same', kernel_initializer='he_normal')(merge6)
    print("conv7 shape:", conv7.shape)
    db7 = denseblock(x=conv7, concat_axis=3, nb_layers=3, growth_rate=16, dropout_rate=0.5)
    print("db7 shape:", db7.shape)
    up7 = Conv2D(128, 2, activation='relu', padding='same', kernel_initializer='he_normal')(
        UpSampling2D(size=(2, 2))(db7))
    print("up7 shape:", up7.shape)
    merge7 = Concatenate(axis=3)([db2, up7])
    print("merge7 shape:", merge7.shape)

    conv8 = Conv2D(128, 3, activation='relu', padding='same', kernel_initializer='he_normal')(merge7)
    print("conv8 shape:", conv8.shape)
    db8 = denseblock(x=conv8, concat_axis=3, nb_layers=3, growth_rate=16, dropout_rate=0.5)
    print("db8 shape:", db8.shape)
    up8 = Conv2D(64, 2, activation='relu', padding='same', kernel_initializer='he_normal')(
        UpSampling2D(size=(2, 2))(db8))
    print("up8 shape:", up8.shape)
    merge8 = Concatenate(axis=3)([db1, up8])
    print("merge8 shape:", merge8.shape)

    conv9 = Conv2D(64, 3, activation='relu', padding='same', kernel_initializer='he_normal')(merge8)
    print("conv9 shape:", conv9.shape)
    db9 = denseblock(x=conv9, concat_axis=3, nb_layers=3, growth_rate=16, dropout_rate=0.5)
    print("db9 shape:", db9.shape)
    conv10 = Conv2D(32, 3, activation='relu', padding='same', kernel_initializer='he_normal')(db9)
    print("conv10 shape:", conv10.shape)
    conv11 = Conv2D(2, 1, activation='softmax')(conv10)
    print("conv11 shape:", conv11.shape)

    model = Model(inputs=inputs, outputs=conv11)

    return model

**Training Configuration: Epochs, Batch Size, and Learning Rate Scheduling**

In [ ]:
batch_size = 16
num_epochs = 120

def lr_schedule(epoch):
    if epoch < 80:
        return 1e-4
    elif epoch < 100:
        return 1e-5
    else:
        return 1e-6

lr_scheduler = LearningRateScheduler(lr_schedule)

In [ ]:
def balanced_cross_entropy(y_true, y_pred, eps=1e-6):
    p = tf.clip_by_value(y_pred, eps, 1.0 - eps)
    loss = - (y_true * tf.math.log(p) + (1.0 - y_true) * tf.math.log(1.0 - p))
    return tf.reduce_mean(loss)


**Custom Metric: Softmax IoU for Binary Object Detection**

In [ ]:
def softmax_iou_metric(y_true, y_pred):
    y_true = K.argmax(y_true, axis=-1)
    y_pred = K.argmax(y_pred, axis=-1)

    y_true = K.cast(y_true, 'float32')
    y_pred = K.cast(y_pred, 'float32')

    intersection = K.sum(y_true * y_pred)
    union = K.sum(y_true) + K.sum(y_pred) - intersection

    iou = intersection / (union + K.epsilon())
    return iou

**Best Epoch Selection Based on Custom Metric**

In [ ]:
def get_best_epoch_by_metric(history, metric_name: str = 'val_softmax_iou_metric', mode: str = 'max') -> int:
    values = history.history.get(metric_name)

    if values is None:
        raise ValueError(f"Metric '{metric_name}' not found in training history.")

    if mode == 'max':
        best_epoch = np.argmax(values) + 1  # epochs start at 1
    elif mode == 'min':
        best_epoch = np.argmin(values) + 1
    else:
        raise ValueError("mode must be either 'max' or 'min'")

    print(f"Best '{metric_name}' at epoch {best_epoch}: {values[best_epoch - 1]:.4f}")
    return best_epoch

**Enhanced Training and Prediction Function for Deep Speckle Correlation**

<div style="font-family: Arial, sans-serif; font-size: 14px;">
    The training function is based on the original function published in the 2018 paper, but several adjustments and improvements have been made.<br>
    The original paper was published several years ago, and therefore, some of the libraries and functions used at that time are no longer relevant in the newer versions of Python and TensorFlow.<br>
    As a result, some of the commands look different and have been adapted to the updated format of the libraries.<br><br>
    1. <b>Updated Data Loading:</b>
        Instead of loading the data as described in the paper, a custom function was used specifically for loading and preprocessing the data.<br>
    2. <b>Additional Evaluation Metrics:</b>
        In addition to the Loss metric used in the original paper, IoU (Intersection over Union) and Accuracy were added to gain a more comprehensive understanding of the model's performance.<br>
    3. <b>Enhanced Model Saving:</b>
        An automatic saving process was implemented for the best model based on the IoU metric, ensuring that less successful models are not saved.<br>
    4. <b>Best Epoch Selection:</b>
        A call to a custom function was included to print the epoch number with the best IoU value.<br>
    5. <b>Performance Visualization:</b>
        Graphs were added to visualize the progression of Loss, IoU, and Accuracy throughout the epochs, providing a visual assessment of the model's performance.<br>
</div>

In [ ]:
def train_and_predict(data_set):

    print('-' * 30)
    print('Loading and preprocessing training and validation sets...')
    print('-' * 30)

    x_train, y_train, x_valid, y_valid = load_processed_train_valid(data_set)

    print('-' * 30)
    print('Creating and compiling model...')
    print('-' * 30)

    model = get_model_deep_speckle()

    def balanced_ce_metric(y_true, y_pred):
      return balanced_cross_entropy(y_true, y_pred)

    model.compile(
        optimizer=Adam(),
        loss=balanced_cross_entropy,
        metrics=['accuracy', softmax_iou_metric],
    )

    model_checkpoint = ModelCheckpoint(
        filepath=save_path + proj_name + '_best.keras',
        monitor='val_softmax_iou_metric',
        mode='max',
        save_best_only=True,
        verbose=2,
        save_freq='epoch',
    )

    print('-' * 30)
    print('Fitting model...')
    print('-' * 30)

    history = model.fit(x_train, y_train, batch_size=batch_size, epochs=num_epochs, verbose=1, shuffle=True,
        callbacks=[model_checkpoint, lr_scheduler], validation_data=(x_valid, y_valid))

    with open(save_path + proj_name + '_history.json', 'w') as f:
        json.dump(history.history, f)

    get_best_epoch_by_metric(history, metric_name='val_softmax_iou_metric', mode='max')

    plt.plot(history.history['loss'])
    plt.plot(history.history['val_loss'])
    plt.title('model loss')
    plt.ylabel('loss')
    plt.xlabel('epoch')
    plt.legend(['train', 'validation'], loc='upper left')
    plt.savefig(save_path + proj_name + '_loss_plot.png')
    plt.show()
    plt.close()

    plt.plot(history.history['softmax_iou_metric'])
    plt.plot(history.history['val_softmax_iou_metric'])
    plt.title('IoU')
    plt.ylabel('IoU')
    plt.xlabel('epoch')
    plt.legend(['train', 'validation'], loc='upper left')
    plt.savefig(save_path + proj_name + '_iou_plot.png')
    plt.show()
    plt.close()

    # Accuracy
    plt.plot(history.history['accuracy'])
    plt.plot(history.history['val_accuracy'])
    plt.title('Accuracy')
    plt.ylabel('accuracy')
    plt.xlabel('epoch')
    plt.legend(['train', 'validation'], loc='upper left')
    plt.savefig(save_path + proj_name + '_acc_plot.png')
    plt.show()
    plt.close()

**Predication Function**

In [ ]:
def predict_testset(
    model_path: str,
    x_test: np.ndarray,
    custom_objects: dict = None,
    verbose: bool = True
) -> np.ndarray:

    if verbose:
        print(f"Loading model from: {model_path}")

    model = load_model(model_path, custom_objects=custom_objects)

    if verbose:
        print(f"Predicting on {len(x_test)} test samples...")

    y_pred = model.predict(x_test, verbose=1)

    if verbose:
        print("Prediction completed.")

    return y_pred

**Prediction Evaluation: IoU and PCC Metrics**

In [ ]:
def evaluate_predictions(y_pred, y_true, threshold: float = 0.5, show_individual: bool = False):
    y_pred_bin = np.argmax(y_pred, axis=-1)
    if y_true.ndim == 4 and y_true.shape[-1] == 2:
        y_true = np.argmax(y_true, axis=-1)

    iou_list = []
    pcc_list = []

    for i in range(len(y_true)):
        y_true_flat = y_true[i].flatten()
        y_pred_flat = y_pred_bin[i].flatten()

        iou = jaccard_score(y_true_flat, y_pred_flat)
        iou_list.append(iou)

        if np.std(y_pred_flat) == 0 or np.std(y_true_flat) == 0:
            pcc = 0
        else:
            pcc, _ = pearsonr(y_true_flat, y_pred_flat)
        pcc_list.append(pcc)

        if show_individual:
            print(f"Image {i}: IoU = {iou:.3f}, PCC = {pcc:.3f}")

    print("\nEvaluation Summary:")
    print(f"Mean IoU (Jaccard): {np.mean(iou_list):.3f}")
    print(f"Mean PCC: {np.mean(pcc_list):.3f}")

**Visualization of Speckle Input, Ground Truth, and Prediction**

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def show_prediction_example(x_test, y_true, y_pred, index=0, threshold: float = 0.5):

    speckle = x_test[index].squeeze()
    gt_mask = np.argmax(y_true[index], axis=-1) if y_true.shape[-1] == 2 else y_true[index]
    pred_mask = np.argmax(y_pred[index], axis=-1)

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 3, 1)
    plt.imshow(speckle, cmap='gray')
    plt.title("Speckle Input")
    plt.axis('off')

    plt.subplot(1, 3, 2)
    plt.imshow(gt_mask, cmap='gray')
    plt.title("Ground Truth")
    plt.axis('off')

    plt.subplot(1, 3, 3)
    plt.imshow(pred_mask, cmap='gray')
    plt.title("Prediction")
    plt.axis('off')

    plt.suptitle(f"Example {index}", fontsize=14)
    plt.tight_layout()
    plt.show()



---



<center><font size="5" color="#2D6A4F"><b>Model 2400</b></font></center>

**Model training**

In [ ]:
proj_name = 'Model2400'  # change this name if retraining
base_ckpt_dir = '/content/drive/MyDrive/TranSpeckle/CNNuNET_checkpoints'

save_path = f"{base_ckpt_dir}/{proj_name}"

In [ ]:
# Optional: Retrain the model from scratch
"""
By default, this notebook uses the provided pre-trained weights.

If you wish to retrain the model, please make sure to update the model name or output directory
to avoid overwriting the existing checkpoint files that were downloaded with the data.
Otherwise, skip the training step and load the pre-trained weights to run prediction and evaluation.
"""
#train_and_predict('2400')


In [ ]:
# Reports the epoch with the best validation performance from the saved training history.

history_file = f"{save_path}/{proj_name}_history.json"

with open(history_file, 'r') as f:
    history_dict = json.load(f)

metric_name = 'val_softmax_iou_metric'
values = history_dict[metric_name]

best_epoch = int(np.argmax(values))
best_value = values[best_epoch]

print(f"Best epoch: {best_epoch + 1} with {metric_name} = {best_value:.4f}")

**group1**

**Load and create predictions**

In [ ]:
x_test1, y_test1 = load_processed_test('group1')

y_pred1 = predict_testset(
    model_path = f"{save_path}/{proj_name}_best.keras",
    x_test=x_test1,
    custom_objects={
        'balanced_cross_entropy': balanced_cross_entropy,
        'softmax_iou_metric': softmax_iou_metric
    }
)

**Evaluation**

In [ ]:
evaluate_predictions(y_pred1, y_test1)

**group2**

**Load and create predictions**

In [ ]:
x_test2, y_test2 = load_processed_test('group2')

y_pred2 = predict_testset(
    model_path = f"{save_path}/{proj_name}_best.keras",
    x_test=x_test2,
    custom_objects={
        'balanced_cross_entropy': balanced_cross_entropy,
        'softmax_iou_metric': softmax_iou_metric
    }
)


**Evaluation**

In [ ]:
evaluate_predictions(y_pred2, y_test2)

**group3**

**Load and create predictions**

In [ ]:
x_test3, y_test3 = load_processed_test('group3')

y_pred3 = predict_testset(
    model_path = f"{save_path}/{proj_name}_best.keras",
    x_test=x_test3,
    custom_objects={
        'balanced_cross_entropy': balanced_cross_entropy,
        'softmax_iou_metric': softmax_iou_metric
    }
)

**Evaluation**

In [ ]:
evaluate_predictions(y_pred3, y_test3)



---



<center><font size="5" color="#2D6A4F"><b>Model1600_4</b></font></center>

**Model training**

In [ ]:
proj_name = 'Model1600_4'  # change this name if retraining
base_ckpt_dir = '/content/drive/MyDrive/TranSpeckle/CNNuNET_checkpoints'

save_path = f"{base_ckpt_dir}/{proj_name}"

In [ ]:
# Optional: Retrain the model from scratch
"""
By default, this notebook uses the provided pre-trained weights.

If you wish to retrain the model, please make sure to update the model name or output directory
to avoid overwriting the existing checkpoint files that were downloaded with the data.
Otherwise, skip the training step and load the pre-trained weights to run prediction and evaluation.
"""
# train_and_predict('1600_4')

In [ ]:
# Reports the epoch with the best validation performance from the saved training history.

history_file = f"{save_path}/{proj_name}_history.json"

with open(history_file, 'r') as f:
    history_dict = json.load(f)

metric_name = 'val_softmax_iou_metric'
values = history_dict[metric_name]

best_epoch = int(np.argmax(values))
best_value = values[best_epoch]

print(f"Best epoch: {best_epoch + 1} with {metric_name} = {best_value:.4f}")

**group 1**

**Load and create predictions**

In [ ]:
y_pred1600_4 = predict_testset(
    model_path = f"{save_path}/{proj_name}_best.keras",
    x_test=x_test1,
    custom_objects={
        'balanced_cross_entropy': balanced_cross_entropy,
        'softmax_iou_metric': softmax_iou_metric
    }
)

**Evaluation**

In [ ]:
evaluate_predictions(y_pred1600_4, y_test1)

<center><font size="5" color="#2D6A4F"><b>Model1600_2</b></font></center>

**Model training**

In [ ]:
proj_name = 'Model1600_2' # change this name if retraining
base_ckpt_dir = '/content/drive/MyDrive/TranSpeckle/CNNuNET_checkpoints'
save_path = f"{base_ckpt_dir}/{proj_name}"

In [ ]:
# Optional: Retrain the model from scratch
"""
By default, this notebook uses the provided pre-trained weights.

If you wish to retrain the model, please make sure to update the model name or output directory
to avoid overwriting the existing checkpoint files that were downloaded with the data.
Otherwise, skip the training step and load the pre-trained weights to run prediction and evaluation.
"""
#train_and_predict('1600_2')

In [ ]:
# Reports the epoch with the best validation performance from the saved training history.

history_file = f"{save_path}/{proj_name}_history.json"

with open(history_file, 'r') as f:
    history_dict = json.load(f)

metric_name = 'val_softmax_iou_metric'
values = history_dict[metric_name]

best_epoch = int(np.argmax(values))
best_value = values[best_epoch]

print(f"Best epoch: {best_epoch + 1} with {metric_name} = {best_value:.4f}")

**group 1**

**Load and create predictions**

In [ ]:
y_pred1600_2 = predict_testset(
    model_path = f"{save_path}/{proj_name}_best.keras",
    x_test=x_test1,
    custom_objects={
        'balanced_cross_entropy': balanced_cross_entropy,
        'softmax_iou_metric': softmax_iou_metric
    }
)

**Evaluation**

In [ ]:
evaluate_predictions(y_pred1600_2, y_test1)

<center><font size="5" color="#2D6A4F"><b>Model800_4</b></font></center>

**Model training**

In [ ]:
proj_name = 'Model800_4' # change this name if retraining
base_ckpt_dir = '/content/drive/MyDrive/TranSpeckle/CNNuNET_checkpoints'
save_path = f"{base_ckpt_dir}/{proj_name}"

In [ ]:
# Optional: Retrain the model from scratch
"""
By default, this notebook uses the provided pre-trained weights.

If you wish to retrain the model, please make sure to update the model name or output directory
to avoid overwriting the existing checkpoint files that were downloaded with the data.
Otherwise, skip the training step and load the pre-trained weights to run prediction and evaluation.
"""
#train_and_predict('800_4')

In [ ]:
# Reports the epoch with the best validation performance from the saved training history.

history_file = f"{save_path}/{proj_name}_history.json"

with open(history_file, 'r') as f:
    history_dict = json.load(f)

metric_name = 'val_softmax_iou_metric'
values = history_dict[metric_name]

best_epoch = int(np.argmax(values))
best_value = values[best_epoch]

print(f"Best epoch: {best_epoch + 1} with {metric_name} = {best_value:.4f}")

**group 1**

**Load and create predictions**

In [ ]:
y_pred800_4 = predict_testset(
    model_path = f"{save_path}/{proj_name}_best.keras",
    x_test=x_test1,
    custom_objects={
        'balanced_cross_entropy': balanced_cross_entropy,
        'softmax_iou_metric': softmax_iou_metric
    }
)

**Evaluation**

In [ ]:
evaluate_predictions(y_pred800_4, y_test1)

<center><font size="5" color="#2D6A4F"><b>Model800_2</b></font></center>

**Model training**

In [ ]:
proj_name = 'Model800_2'# change this name if retraining
base_ckpt_dir = '/content/drive/MyDrive/TranSpeckle/CNNuNET_checkpoints'
save_path = f"{base_ckpt_dir}/{proj_name}"

In [ ]:
# Optional: Retrain the model from scratch
"""
By default, this notebook uses the provided pre-trained weights.

If you wish to retrain the model, please make sure to update the model name or output directory
to avoid overwriting the existing checkpoint files that were downloaded with the data.
Otherwise, skip the training step and load the pre-trained weights to run prediction and evaluation.
"""
#train_and_predict('800_2')

In [ ]:
# Reports the epoch with the best validation performance from the saved training history.

history_file = f"{save_path}/{proj_name}_history.json"

with open(history_file, 'r') as f:
    history_dict = json.load(f)

metric_name = 'val_softmax_iou_metric'
values = history_dict[metric_name]

best_epoch = int(np.argmax(values))
best_value = values[best_epoch]

print(f"Best epoch: {best_epoch + 1} with {metric_name} = {best_value:.4f}")

**group 1**

**Load and create predictions**

In [ ]:
y_pred800_2 = predict_testset(
    model_path = f"{save_path}/{proj_name}_best.keras",
    x_test=x_test1,
    custom_objects={
        'balanced_cross_entropy': balanced_cross_entropy,
        'softmax_iou_metric': softmax_iou_metric
    }
)

**Evaluation**

In [ ]:
evaluate_predictions(y_pred800_2, y_test1)

<center><font size="5" color="#2D6A4F"><b>Model800_1</b></font></center>

**Model training**

In [ ]:
proj_name = 'Mode800_1' # change this name if retraining
base_ckpt_dir = '/content/drive/MyDrive/TranSpeckle/CNNuNET_checkpoints'
save_path = f"{base_ckpt_dir}/{proj_name}"

In [ ]:
# Optional: Retrain the model from scratch
"""
By default, this notebook uses the provided pre-trained weights.

If you wish to retrain the model, please make sure to update the model name or output directory
to avoid overwriting the existing checkpoint files that were downloaded with the data.
Otherwise, skip the training step and load the pre-trained weights to run prediction and evaluation.
"""
#train_and_predict('800_1')

In [ ]:
# Reports the epoch with the best validation performance from the saved training history.

history_file = f"{save_path}/{proj_name}_history.json"

with open(history_file, 'r') as f:
    history_dict = json.load(f)

metric_name = 'val_softmax_iou_metric'
values = history_dict[metric_name]

best_epoch = int(np.argmax(values))
best_value = values[best_epoch]

print(f"Best epoch: {best_epoch + 1} with {metric_name} = {best_value:.4f}")

**group 1**

**Load and create predictions**

In [ ]:
y_pred800_1 = predict_testset(
    model_path = f"{save_path}/{proj_name}_best.keras",
    x_test=x_test1,
    custom_objects={
        'balanced_cross_entropy': balanced_cross_entropy,
        'softmax_iou_metric': softmax_iou_metric
    }
)

**Evaluation**

In [ ]:
evaluate_predictions(y_pred800_1, y_test1)

**group4_1**

**Load and create predictions**

In [ ]:
x_test4_1, y_test4_1 = load_processed_test('group4_1')


y_pred4_1 = predict_testset(
    model_path = f"{save_path}/{proj_name}_best.keras",
    x_test=x_test4_1,
    custom_objects={
        'balanced_cross_entropy': balanced_cross_entropy,
        'softmax_iou_metric': softmax_iou_metric
    }
)

**Evaluation**

In [ ]:
evaluate_predictions(y_pred4_1, y_test4_1)

**group4_1**

**Load and create predictions**

In [ ]:
x_test4_2, y_test4_2 = load_processed_test('group4_2')


y_pred4_2 = predict_testset(
    model_path = f"{save_path}/{proj_name}_best.keras",
    x_test=x_test4_2,
    custom_objects={
        'balanced_cross_entropy': balanced_cross_entropy,
        'softmax_iou_metric': softmax_iou_metric
    }
)

**Evaluation**

In [ ]:
evaluate_predictions(y_pred4_2, y_test4_2)